# 02 · 物理资产构建
读取模块一导入的 cabinet2 USD，构建真实凸碰撞体、显式物性先验及可回读的标准 USD 资产。

物性先验用于仿真和训练开发，来源、假设与接触约束随资产保存；独立实测物性和真机验证仍属于不同验收结论。全部使用当前 PhysReal-X Conda 环境。

In [ ]:
import json
import os
import sys
import uuid
from pathlib import Path
from tempfile import TemporaryDirectory

os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("OMP_NUM_THREADS", "4")
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/physrealx").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if Path(sys.prefix).name != "PhysReal-X":
    raise RuntimeError("请选择 PhysReal-X Conda 环境的 Jupyter 内核")
from examples._common import (
    NotebookRun,
    operation_receipt,
    output_artifact,
    require_completed,
)

from physrealx import ArtifactRef, AssetSpec, PhysRealX

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
OUTPUT_DIR = PROJECT_ROOT / "examples/physical_asset_building/output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run = NotebookRun(
    OUTPUT_DIR / "report.json",
    ("input", "collision", "assembly", "package", "quality", "export"),
    {"python": sys.executable, "conda_prefix": sys.prefix},
    evidence_scope="supplied_usd_with_explicit_physics_priors",
)
run_key = uuid.uuid4().hex[:12]
print(sys.executable)

In [ ]:
with run.stage("input") as stage:
    upstream = require_completed(
        PROJECT_ROOT / "examples/real_assets_and_digital_twin/output/digital-twin-report.json"
    )
    reference_model = ArtifactRef.model_validate(
        upstream["stages"]["export"]["artifacts"]["reference-model.json"]
    )
    reference = json.loads(px.artifacts.get_bytes(reference_model))
    stage["reference_model"] = reference_model
    print(reference["root_link"], [link["name"] for link in reference["links"]])
    reference_registration = ArtifactRef.model_validate(
        upstream["stages"]["export"]["artifacts"]["reference-registration.json"]
    )

In [ ]:
with run.stage("collision") as stage:
    collision_reports = []
    receipts = []
    for link in reference["links"]:
        operation = px.assets.build_collision(
            ArtifactRef.model_validate(link["geometry"]),
            method="coacd",
            config={
                "source_unit": "m",
                "link": link["name"],
                "threshold_m": 0.003,
                "max_parts": 128,
                "max_volume_error": 0.15,
                "preprocessing": "coacd",
            },
        )
        collision_reports.append(output_artifact(operation, "collision_build_report"))
        receipts.append(operation_receipt(operation))
        print(link["name"], dict(operation.metrics))
    stage["operations"] = receipts

In [ ]:
with run.stage("assembly") as stage:
    # 所有未测量参数均显式列出；来源为 default，置信度为 0，不升级为实测资产。
    # 导轨摩擦由关节模型表达，相邻链接不重复施加接触摩擦；外部物体碰撞仍开启。
    physics_prior = {
        "density_kg_m3": 650.0,
        "static_friction": 0.3,
        "dynamic_friction": 0.2,
        "restitution": 0.0,
        "joint_damping": 1.0,
        "joint_friction": 0.1,
        "effort_limit": 40.0,
        "joint_models_adjacent_contact": True,
    }
    assembly_input = px.artifacts.put_json(
        {
            "schema": "physrealx.reference-assembly-input/1",
            "resource_id": f"cabinet2.physical.{run_key}",
            "version": "1.0.0",
            "display_name": "Cabinet2 supplied USD with explicit physics priors",
            "reference_model": reference_model.model_dump(mode="json"),
            "registration": reference_registration.model_dump(mode="json"),
            "collision_reports": [ref.model_dump(mode="json") for ref in collision_reports],
            "physics_prior": physics_prior,
        }
    )
    assembly_operation = px.assets.assemble(assembly_input, method="reference_asset_assembly")
    assembly = output_artifact(assembly_operation, "asset_assembly")
    asset = AssetSpec.model_validate(json.loads(px.artifacts.get_bytes(assembly))["asset_spec"])
    stage["operation"] = operation_receipt(assembly_operation)
    print("物性证据状态:", asset.evidence_state.value)
    print("各部件先验质量 kg:", {body.link: body.mass_kg.value for body in asset.physics})

In [ ]:
with run.stage("package") as stage:
    package_operation = px.assets.package(assembly, method="openusd")
    usd = output_artifact(package_operation, "usd_asset_bundle")
    package_manifest = output_artifact(package_operation, "asset_package_manifest")
    stage["operation"] = operation_receipt(package_operation)
    print("USD 数值回读:", dict(package_operation.metrics))

In [ ]:
with run.stage("quality") as stage:
    quality = px.assets.validate(asset, profile="simulation")
    assert quality.passed, quality.model_dump(mode="json")
    asset_ref = px.assets.register(asset)
    stage["asset"] = asset_ref
    stage["quality"] = quality
    print("可仿真资产:", asset_ref.resource_id, "关节:", len(asset.joints))
    print("接触约束:", asset.contact_exclusions)

In [ ]:
with run.stage("export") as stage:
    exports = {"cabinet2.usdz": usd, "assembly.json": assembly, "package.json": package_manifest}
    with TemporaryDirectory(prefix=".export-", dir=OUTPUT_DIR) as directory:
        for filename, artifact in exports.items():
            px.artifacts.materialize(artifact, Path(directory) / filename)
        for filename in exports:
            (Path(directory) / filename).replace(OUTPUT_DIR / filename)
    stage["artifacts"] = exports
    stage["asset"] = asset_ref
    stage["source_reference"] = reference_model
report = run.complete()
print(report["execution_status"], OUTPUT_DIR / "report.json")